# LeWM modules

> Building blocks of LeWM: image encoder, action embedder, conditional transformer predictor.

In [ ]:
#| default_exp wm.lewm.module

In [ ]:
#| hide
from nbdev.showdoc import *

The predictor and its blocks are stable-worldmodel's. The image encoder is a plain-torch ViT
returning the CLS embedding (stable-worldmodel loads a Hugging Face ViT), and
`ImagePreprocessor` turns raw ``uint8`` channel-last images (e.g. the ``pov`` observations) into
the normalised channel-first input, inside the model.

In [ ]:
#| export
# Adapted from stable-worldmodel's world models (`wm/`) (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

In [ ]:
#| export
def modulate(x, shift, scale):
    "AdaLN-zero modulation."
    return x * (1 + scale) + shift


class FeedForward(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(hidden_dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    "Multi-head scaled dot-product attention, causal by default."
    def __init__(self, dim: int, heads: int = 8, dim_head: int = 64, dropout: float = 0.0):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads, self.dropout = heads, dropout
        self.norm = nn.LayerNorm(dim)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = (nn.Sequential(nn.Linear(inner_dim, dim), nn.Dropout(dropout))
                       if not (heads == 1 and dim_head == dim) else nn.Identity())

    def forward(self, x, causal: bool = True):
        B, T, _ = x.shape
        q, k, v = (t.reshape(B, T, self.heads, -1).transpose(1, 2) for t in self.to_qkv(self.norm(x)).chunk(3, dim=-1))
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=causal)
        return self.to_out(out.transpose(1, 2).reshape(B, T, -1))


class ConditionalBlock(nn.Module):
    "Causal transformer block conditioned (AdaLN-zero) on a per-token vector, e.g. the action embedding."
    def __init__(self, dim: int, heads: int, dim_head: int, mlp_dim: int, dropout: float = 0.0):
        super().__init__()
        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim, bias=True))
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6, dim=-1)
        x = x + gate_msa * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))
        return x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))


class Block(nn.Module):
    "Transformer block (causal unless ``causal=False``)."
    def __init__(self, dim: int, heads: int, dim_head: int, mlp_dim: int, dropout: float = 0.0, causal: bool = True):
        super().__init__()
        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        self.mlp = FeedForward(dim, mlp_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.causal = causal

    def forward(self, x):
        x = x + self.attn(self.norm1(x), causal=self.causal)
        return x + self.mlp(self.norm2(x))


class Transformer(nn.Module):
    "Stack of transformer blocks (conditional or not), with input / output projections."
    def __init__(self, input_dim, hidden_dim, output_dim, depth, heads, dim_head, mlp_dim, dropout=0.0,
                 block_class=Block):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.input_proj = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()
        self.cond_proj = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()
        self.output_proj = nn.Linear(hidden_dim, output_dim) if hidden_dim != output_dim else nn.Identity()
        self.layers = nn.ModuleList([block_class(hidden_dim, heads, dim_head, mlp_dim, dropout) for _ in range(depth)])

    def forward(self, x, c=None):
        x = self.input_proj(x)
        if c is not None:
            c = self.cond_proj(c)
        for block in self.layers:
            x = block(x) if isinstance(block, Block) else block(x, c)
        return self.output_proj(self.norm(x))

In [ ]:
#| export
class Embedder(nn.Module):
    "Action embedder: ``(B, T, input_dim)`` -> ``(B, T, emb_dim)``."
    def __init__(self, input_dim: int = 10, smoothed_dim: int = 10, emb_dim: int = 10, mlp_scale: int = 4):
        super().__init__()
        self.input_dim, self.smoothed_dim, self.emb_dim = input_dim, smoothed_dim, emb_dim
        self.patch_embed = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1, stride=1)
        self.embed = nn.Sequential(nn.Linear(smoothed_dim, mlp_scale * emb_dim), nn.SiLU(),
                                   nn.Linear(mlp_scale * emb_dim, emb_dim))

    def forward(self, x):
        x = self.patch_embed(x.to(self.patch_embed.weight.dtype).permute(0, 2, 1)).permute(0, 2, 1)
        return self.embed(x)


class MLP(nn.Module):
    "Linear -> norm -> activation -> linear, on ``(N, input_dim)``."
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int | None = None, norm_fn=nn.LayerNorm,
                 act_fn=nn.GELU):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim),
                                 norm_fn(hidden_dim) if norm_fn is not None else nn.Identity(), act_fn(),
                                 nn.Linear(hidden_dim, output_dim or input_dim))

    def forward(self, x):
        return self.net(x)


class Predictor(nn.Module):
    "Autoregressive predictor of the next embeddings, ``(B, T, d)`` conditioned on ``(B, T, act_dim)``."
    def __init__(self, *, num_frames, depth, heads, mlp_dim, input_dim, hidden_dim, output_dim=None, dim_head=64,
                 dropout=0.0, emb_dropout=0.0):
        super().__init__()
        self.num_frames, self.input_dim, self.hidden_dim = num_frames, input_dim, hidden_dim
        self.output_dim = output_dim or input_dim
        self.pos_embedding = nn.Parameter(torch.randn(1, num_frames, input_dim))
        self.dropout = nn.Dropout(emb_dropout)
        self.transformer = Transformer(input_dim, hidden_dim, output_dim or input_dim, depth, heads, dim_head,
                                       mlp_dim, dropout, block_class=ConditionalBlock)

    def forward(self, x, c):
        x = self.dropout(x + self.pos_embedding[:, :x.size(1)])
        return self.transformer(x, c)

In [ ]:
#| export
VIT_SCALES = {   # (dim, depth, heads, mlp_dim)
    'tiny': (192, 12, 3, 768),
    'small': (384, 12, 6, 1536),
    'base': (768, 12, 12, 3072),
}

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class ViTEncoder(nn.Module):
    "Vision transformer: images ``(N, C, H, W)`` -> CLS embedding ``(N, dim)``."
    def __init__(self, image_size: int = 224, patch_size: int = 14, dim: int = 192, depth: int = 12, heads: int = 3,
                 mlp_dim: int = 768, channels: int = 3, dropout: float = 0.0):
        super().__init__()
        if image_size % patch_size:
            raise ValueError(f"image_size {image_size} must be a multiple of patch_size {patch_size}")
        self.dim = dim
        self.patch_embed = nn.Conv2d(channels, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, (image_size // patch_size) ** 2 + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, dim // heads, mlp_dim, dropout, causal=False) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    @classmethod
    def from_scale(cls, scale: str = 'tiny', **kwargs) -> ViTEncoder:
        "A ViT of a standard size ('tiny', 'small', 'base')."
        dim, depth, heads, mlp_dim = VIT_SCALES[scale]
        return cls(**{'dim': dim, 'depth': depth, 'heads': heads, 'mlp_dim': mlp_dim, **kwargs})

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], dim=1) + self.pos_embedding[:, :x.shape[1] + 1]
        for block in self.blocks:
            x = block(x)
        return self.norm(x)[:, 0]


class ExtraEncoder(nn.Module):
    "Small MLP encoding a vector input (e.g. the agent's ``position``) multiplied by `scale`: ``(N, input_dim)`` -> ``(N, dim)``."
    def __init__(self, input_dim: int, dim: int = 32, hidden_dim: int = 64, scale: float = 1.0):
        super().__init__()
        self.scale = scale
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, dim))
        self.dim = dim

    def forward(self, x):
        return self.net(x.float() * self.scale)


class ImagePreprocessor(nn.Module):
    """
    Raw images ``(..., H, W, C)`` (``uint8``, or floats in [0, 255]) -> normalised floats
    ``(..., C, image_size, image_size)``: resized (bilinear, antialiased), ImageNet statistics, as
    in stable-worldmodel's LeWM training.
    """
    def __init__(self, image_size: int = 224, mean=IMAGENET_MEAN, std=IMAGENET_STD):
        super().__init__()
        self.image_size = image_size
        self.register_buffer('mean', torch.tensor(mean).view(1, -1, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, -1, 1, 1))

    def forward(self, x):
        lead, (H, W, C) = x.shape[:-3], x.shape[-3:]
        x = x.reshape(-1, H, W, C).permute(0, 3, 1, 2).to(self.mean.dtype) / 255.0
        if (H, W) != (self.image_size, self.image_size):
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', antialias=True,
                              align_corners=False)
        x = (x - self.mean) / self.std
        return x.reshape(*lead, C, self.image_size, self.image_size)

### Tests

In [ ]:
enc = ViTEncoder(image_size=32, patch_size=8, dim=32, depth=2, heads=2, mlp_dim=64)
pre = ImagePreprocessor(32)
images = torch.randint(0, 256, (5, 56, 56, 3), dtype=torch.uint8)
assert pre(images).shape == (5, 3, 32, 32) and enc(pre(images)).shape == (5, 32)
pred = Predictor(num_frames=3, depth=2, heads=2, mlp_dim=64, input_dim=32, hidden_dim=32, dim_head=16)
x, c = torch.randn(4, 3, 32), torch.randn(4, 3, 32)
out = pred(x, c)
assert out.shape == (4, 3, 32)
x2 = x.clone(); x2[:, 2] += 1
assert torch.allclose(pred(x2, c)[:, :2], out[:, :2], atol=1e-5)   # causal: the past ignores the future
assert Embedder(input_dim=7, emb_dim=32)(torch.randn(4, 3, 7)).shape == (4, 3, 32)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()